# 04 — 10-fold baseline comparison and synthetic-noise evaluation

All classical baselines use exactly the same 10 speaker-grouped folds as the CNN. Cross-validation metrics are summarized across folds. Each baseline is then fitted on the complete outer training partition and evaluated on the same untouched final test speakers. The final CNN from Notebook 03 is evaluated on that identical test partition.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
import tensorflow as tf
import joblib

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

OUT = Path("anu_revision_output")
data = np.load(OUT / "speaker_split_mfcc_10fold.npz", allow_pickle=True)

X_train = data["X_train"]
X_test = data["X_test"]
y_train = data["y_train"]
y_test = data["y_test"]
fold_assignments = data["fold_assignments"]
classes = data["classes"].astype(str)

X_train_flat = X_train.reshape(len(X_train), -1)
X_test_flat = X_test.reshape(len(X_test), -1)

print("Training samples:", len(y_train))
print("Final test samples:", len(y_test))
print("Folds:", sorted(np.unique(fold_assignments).tolist()))

Training samples: 3522
Final test samples: 918
Folds: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


In [2]:
def create_baselines():
    return {
        "SVM (RBF)": SVC(kernel="rbf", class_weight="balanced"),
        "Random Forest": RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1
        ),
        "KNN (5-NN)": KNeighborsClassifier(n_neighbors=5)
    }

def calculate_metrics(y_true, y_pred):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro precision": precision,
        "Macro recall": recall,
        "Macro F1": f1
    }

In [3]:
# Ten-fold cross-validation for each classical baseline.
cv_rows = []

for model_name in ["SVM (RBF)", "Random Forest", "KNN (5-NN)"]:
    print(f"Running 10-fold CV: {model_name}")

    for fold in range(1, 11):
        fit_mask = fold_assignments != fold
        val_mask = fold_assignments == fold

        model = create_baselines()[model_name]
        model.fit(X_train_flat[fit_mask], y_train[fit_mask])
        predictions = model.predict(X_train_flat[val_mask])

        metrics = calculate_metrics(y_train[val_mask], predictions)
        cv_rows.append({
            "Model": model_name,
            "Fold": fold,
            **metrics
        })

baseline_cv = pd.DataFrame(cv_rows)
display(baseline_cv)
baseline_cv.to_csv(OUT / "baseline_10fold_fold_results.csv", index=False)

baseline_cv_summary = baseline_cv.groupby("Model")[
    ["Accuracy", "Macro precision", "Macro recall", "Macro F1"]
].agg(["mean", "std"])
display(baseline_cv_summary)
baseline_cv_summary.to_csv(OUT / "baseline_10fold_summary.csv")

Running 10-fold CV: SVM (RBF)
Running 10-fold CV: Random Forest
Running 10-fold CV: KNN (5-NN)


,Model,Fold,Accuracy,Macro precision,Macro recall,Macro F1
0,SVM (RBF),1,0.189655,0.246088,0.194643,0.151648
1,SVM (RBF),2,0.222222,0.258444,0.223214,0.214884
2,SVM (RBF),3,0.178788,0.269843,0.180556,0.188916
3,SVM (RBF),4,0.224242,0.230014,0.223214,0.202963
4,SVM (RBF),5,0.309942,0.312544,0.304167,0.292113
5,SVM (RBF),6,0.190909,0.243315,0.187500,0.161108
6,SVM (RBF),7,0.298246,0.429461,0.297619,0.314985
7,SVM (RBF),8,0.236715,0.244000,0.236905,0.226139
8,SVM (RBF),9,0.220238,0.216891,0.220238,0.207049
9,SVM (RBF),10,0.255952,0.304219,0.255952,0.255422


Accuracy           Macro precision           Macro recall  \
                   mean       std            mean       std         mean   
Model                                                                      
KNN (5-NN)     0.251069  0.065276        0.286002  0.085072     0.251389   
Random Forest  0.346251  0.075872        0.384701  0.072247     0.345496   
SVM (RBF)      0.232691  0.044299        0.275482  0.062023     0.232401   

                         Macro F1            
                    std      mean       std  
Model                                        
KNN (5-NN)     0.065640  0.248809  0.072786  
Random Forest  0.074996  0.339526  0.074415  
SVM (RBF)      0.042720  0.221523  0.052791

In [4]:
# Fit each classical baseline on all outer-training speakers and evaluate once on final test.
final_rows = []

for model_name, model in create_baselines().items():
    model.fit(X_train_flat, y_train)
    predictions = model.predict(X_test_flat)
    metrics = calculate_metrics(y_test, predictions)

    final_rows.append({"Model": model_name, **metrics})
    joblib.dump(
        model,
        OUT / (model_name.lower().replace(" ", "_").replace("(", "").replace(")", "") + "_10fold_final.joblib")
    )

# Add final CNN results from Notebook 03.
cnn_path = OUT / "anu_cnn_10fold_final.keras"
cnn = tf.keras.models.load_model(cnn_path)
cnn_predictions = cnn.predict(X_test, verbose=0).argmax(axis=1)
cnn_metrics = calculate_metrics(y_test, cnn_predictions)
final_rows.append({"Model": "CNN", **cnn_metrics})

final_comparison = pd.DataFrame(final_rows).sort_values("Accuracy", ascending=False)
display(final_comparison)
final_comparison.to_csv(OUT / "baseline_10fold_final_test_comparison.csv", index=False)

,Model,Accuracy,Macro precision,Macro recall,Macro F1
1,Random Forest,0.386710,0.405551,0.386905,0.377821
0,SVM (RBF),0.287582,0.310406,0.287879,0.286879
2,KNN (5-NN),0.285403,0.308426,0.285606,0.291103
3,CNN,0.263617,0.524371,0.262121,0.213240


In [5]:
# Synthetic Gaussian-noise test on the same final test audio paths.
SR = 22050
N_MFCC = 20
FRAMES = 40
mean = float(data["mean"])
std = float(data["std"])
test_paths = data["test_paths"]

def extract_noisy_mfcc(path, seed):
    signal, _ = librosa.load(path, sr=SR, mono=True)
    rng = np.random.default_rng(seed)
    noisy_signal = signal + rng.normal(0, 0.005, size=len(signal))

    features = librosa.feature.mfcc(
        y=noisy_signal,
        sr=SR,
        n_mfcc=N_MFCC
    )

    if features.shape[1] < FRAMES:
        features = np.pad(features, ((0, 0), (0, FRAMES - features.shape[1])))
    else:
        features = features[:, :FRAMES]

    return ((features.astype("float32") - mean) / std)[..., None]

X_noise = np.stack([
    extract_noisy_mfcc(path, seed=42 + index)
    for index, path in enumerate(test_paths)
])

clean_predictions = cnn.predict(X_test, verbose=0).argmax(axis=1)
noise_predictions = cnn.predict(X_noise, verbose=0).argmax(axis=1)

noise_results = pd.DataFrame([
    {"Condition": "Clean", **calculate_metrics(y_test, clean_predictions)},
    {"Condition": "Synthetic Gaussian noise (std=0.005)", **calculate_metrics(y_test, noise_predictions)}
])

display(noise_results)
noise_results.to_csv(OUT / "noise_10fold_final_test_results.csv", index=False)

,Condition,Accuracy,Macro precision,Macro recall,Macro F1
0,Clean,0.263617,0.524371,0.262121,0.213240
1,Synthetic Gaussian noise (std=0.005),0.342048,0.523845,0.342316,0.291486


### Outputs

- `baseline_10fold_fold_results.csv` — per-fold baseline metrics.
- `baseline_10fold_summary.csv` — mean and standard deviation across folds.
- `baseline_10fold_final_test_comparison.csv` — final test comparison of CNN, SVM, Random Forest, and KNN.
- `noise_10fold_final_test_results.csv` — clean versus synthetic-noise final test metrics.

Use the fold-summary table to compare cross-validation performance. Use the final-test comparison as a separate held-out evaluation.